# 9주차 실습 — 분석 질문을 SQL로 바꾸기

- **데이터**: `aact_mini` — 임상시험 37,509건(2015~2024년 시작). 7 · 8주차와 같은 데이터입니다.
- **오늘 배우는 것**: ① SQL 분석을 위해 질문을 다섯 단계로 나누는 팁 · ② 서브쿼리 이용하기 · ③ 조건을 하나씩 더하며 결과를 확인하기 · ④ 검산으로 스스로 점검하기
- 이 실습은 점수가 없습니다.

## 0. 시작하기 전에

- Colab 에서는 먼저 **`File → Save a copy in Drive`** 를 누르세요.
- 셀은 **위에서부터 차례로** 실행합니다 (▶ 또는 `Shift + Enter`). `%%sql` 아래의 `-- Q1` · `-- S1` 같은 문항 이름 줄은 지우지 마세요.
- 문항 SQL 을 실행하면 결과 아래에 **✅ / ❌** 가 바로 나옵니다.
- 오류가 나면 **`… Error:` 로 시작하는 줄**만 읽으면 됩니다.

In [ ]:
#@title [1번] ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
_pd.set_option("display.max_columns", None)     # SELECT * 로 열 이름을 볼 때 모든 열을 보여 줌
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "↓ 아래 '… Error:' 로 시작하는 줄이 원인입니다", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게

class _SameCursor:                           # jupysql 이 셀마다 새 cursor(=새 연결)를 열면
    def __init__(self, c): self._c = c       # CREATE TEMP TABLE 로 만든 표가 다음 셀에서 사라지므로
    def close(self): pass                    # 항상 같은 연결을 쓰게 합니다
    def __getattr__(self, k): return getattr(self._c, k)

class _OneConnection:
    def __init__(self, c): self._c = c
    def cursor(self): return _SameCursor(self._c)
    def __getattr__(self, k): return getattr(self._c, k)

_aact = _OneConnection(conn)
%sql _aact --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt
import re as _re, zlib as _zlib, base64 as _b64, json as _json

def _load_answers(blob):
    """자동 확인용 지문(정답 결과표의 해시 · 행 수 · 열 이름 · 힌트)을 풉니다. 정답 SQL 은 들어 있지 않습니다."""
    return _json.loads(_zlib.decompress(_b64.b64decode(blob)).decode())

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False, "sql": ""}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            if parts and _re.match(r"^\[\d+번\]$", parts[0]):      # 수업용 셀 번호 [12번] 은 건너뜀
                parts = parts[1:]
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    body = "\n".join(l.split("--")[0] for l in src.split("\n")[1:])
    _LAST.update(tag=tag, error=not result.success, sql=body)
    if tag and tag in globals().get("ANSWERS", {}):         # 문항 셀(-- Q1)이면 실행하자마자 ✅/❌ 표시
        check(tag, getattr(result, "result", None))

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 분석 질환 정보가 없습니다. 맨 위 준비 셀을 다시 실행하세요."); return
    if _LAST["sql"] and not _re.search(r"\bFROM\b", _LAST["sql"], _re.I):
        return _miss(name, "FROM 이 없는 SQL 입니다. 숫자를 직접 적지 말고, 표에서 꺼내 세는 SQL 을 쓰세요.", exp)
    if _LAST["error"]:
        return _miss(name, "SQL 에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, "결과가 표가 아닙니다. SQL 이 SELECT 로 끝나는지 확인하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 실행 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

print("연결 완료:", DB)

ANSWERS = _load_answers("eNqtlF9L21AYxr/KS64qyGiaf+3YBiJluyiCuD8XY0ibprMgVqywCxHamUHXdtiqnSlrnEI3newi0zgV+mm8zHvyHfaeGmdSGKxuV8l5zsk57+953pM1YVYU7sOaUC6+pqcgFzTJUOKpXCKlC5MglP6gr5TelEkX6VUvLfLXlwJr9Pxdi9Us4RXJC8WlVf6liI0j79SEudkMsBMLPw8m4apiea7pWxfAOjXA4294YLPdFuB2D+s74J1sXFW6zDbBO93HLy7X8LDFhSV9db6YB9zq8/Vsp4qNGjb694a1ruSNFSNPhxayi2WDK8urxdJSdvFGWidtNhHmFfWEauQShqiK2TCvLiWTSjyfleSCGOJNhXmvwfjsLTkf2BfsU2ue7dWY3YxYMZfOpKefwrDybReYPUDHoqEbCzx6CMsL2bIxgQc9wO8/Wf1sfDwpiifrSS2nKlJCCuON6gFe8p/w8LIGnrMVm3mWyUwwu0I4+NaCB49gyHNpemctv2OxwwrFXr+FA+n6KGZbvDOCbZhNDnVMbDSpF4AdON4P1zsd0FT3Tr7IYV+MnKpruiarqWjso/o4vvTbrNOlwiOe/FbB7xzRSt7KcvhG8F4gK2+IIOZvvvfO97l9U88fT4xDqIQJC2pKU428mtSVCOGoPgYhNiveeQ+PjyKElJgSj+PxO+pnl21Uh2lR9dcibpp0q/vso+s57Zu8lcCA6am5NLx4kp75D/GqkXglVZI1VVeSmhaJd0QP4JU7w1N/oz3g5PjV5NT4waGBbzrDYPer/l4TmFUl/mAReOcXtBfdiEkKvk3f4I7DLNdv9/jPMPJXg5ga3IxuZWiMzR9bf90U678Af8l9UQ==")   # 자동 확인용 (정답 SQL 은 없음)

## 1. 질문을 SQL로 바꾸기 (팁) — 다섯 단계 절차

### 분석 질문을 SQL로 바꾸는 절차 — 다섯 단계를 순서대로

| 단계 | 할 일 |
|---|---|
| ① 질문을 한 문장으로 | “대상, 기간, 조건” 등을 분명히 적는다 (SELECT, FROM, WHERE) |
| ② 세는 단위를 정한다 | 시험 건수? 환자 수? — 결과의 한 행이 무엇인가 |
| ③ 필요한 표와 연결 키 | 어느 표에 있는 값인가, 무엇으로 연결하는가 |
| ④ 조건을 하나씩 더한다 | 한 번에 다 쓰지 않고 건수 변화를 보며 SQL 식을 추가 |
| ⑤ 검산한다 | 건수 · 분모 · 중복을 확인한 뒤 결과를 쓴다 |

### 서브쿼리와 JOIN

| | 서브쿼리 (IN) | JOIN |
|---|---|---|
| 무엇을 하나 | 대상 목록을 만들어 걸러냄 | 두 표를 옆으로 이어 붙임 |
| 행 수가 늘어나나 | 늘지 않음 (원래 표 그대로) | 1 : N 이면 늘어남 |
| 언제 쓰나 | ‘이 조건에 맞는 대상만’ 고를 때 | 다른 표의 **열을 함께 보여줄** 때 |
| 이번 사례에서는 | 대상 시험 고르기 | 상태 · 국가를 함께 보기 |

대상을 고르기만 할 때는 서브쿼리가 안전합니다. (**행 수가 변하지 않음**)

### 서브쿼리 — 대상을 먼저 뽑고, 그 안에서 세기

괄호 안이 먼저 계산됩니다.

괄호 안에서 ‘대상자 목록’을 먼저 만들고, 바깥에서 그 목록에 드는 것만 셉니다.

In [ ]:
%%sql
-- [2번] “제2형 당뇨 대상 시험이 몇 건이야?” → 1,353
SELECT COUNT(DISTINCT nct_id) AS 시험수
FROM studies
WHERE nct_id IN (
    SELECT nct_id
    FROM browse_conditions
    WHERE mesh_term = 'Diabetes Mellitus, Type 2'
);

## 2. 종합 예제 — 데이터 파악부터 보고까지

제2형 당뇨병 시험으로 분석 순서 0~7단계를 따라 갑니다.

### 0단계 — 데이터 파악하기

**질문** 어느 표의 어느 열에 질환 · 임상 단계 · 등록 인원이 들어 있나?

결과가 보이게 `SELECT * FROM ( )` 로 감쌉니다.

In [ ]:
%%sql
-- [3번] 표 14개: studies, browse_conditions …
SELECT * FROM (SHOW TABLES);

In [ ]:
%%sql
-- [4번] 열 이름과 타입을 보여 줌
SELECT * FROM (DESCRIBE studies);

In [ ]:
%%sql
-- [5번] 실제 값의 모양 확인 (예: 'PHASE3')
SELECT * FROM studies LIMIT 5;

**DESCRIBE studies 결과 (일부)**

| column_name | column_type | 뜻 |
|---|---|---|
| nct_id | VARCHAR | 시험 번호 (키) |
| phase | VARCHAR | 임상 단계 |
| overall_status | VARCHAR | 진행 상태 |
| enrollment | BIGINT | 등록 인원 |
| results_first_posted_date | DATE | 결과 게시일 |

**알게 된 것** 질환은 `browse_conditions.mesh_term`, 나머지는 `studies` 에 있고 `nct_id` 로 연결됨. `studies` 는 시험 1건, `browse_conditions` 는 시험 × 질환 1쌍임.

### 0단계 — 질환 이름 찾기

**질문** 제2형 당뇨병은 mesh_term 에 어떤 이름으로 적혀 있나?

**찾은 것** 제2형 당뇨병은 `'Diabetes Mellitus, Type 2'` (1,353건). 1단계부터 이 이름을 그대로 씀

**찾기 vs 고르기** 찾을 때는 `ILIKE '%…%'`, 고를 때는 `=` 로 정확한 이름 하나 (ILIKE 로 고르면 제1형 등이 섞여 2,078건)

**상위 용어** Diabetes Mellitus(2,064건)는 제1형 · 제2형 등을 묶는 넓은 이름

In [ ]:
%%sql
-- [6번] 질환 이름 찾기 → 12행 (Diabetes Mellitus 2,064 · Type 2 1,353 · Type 1 216 …)
SELECT mesh_term, COUNT(*) AS 시험수
FROM browse_conditions
WHERE mesh_term ILIKE '%diabetes%'
GROUP BY mesh_term
ORDER BY 시험수 DESC;

### 0단계 — 나라 이름 찾기

**질문** 한국은 countries 표에 어떤 이름으로 적혀 있나? (6단계에서 씀)

**찾은 것** 한국은 `'South Korea'` 하나. `'Korea'` 로 쓰면 0건

**removed** true = 등록 기록에서 빠진 국가 (28행). 6단계에서 `removed = false` 로 뺌

In [ ]:
%%sql
-- [7번] 나라 이름 찾기 → South Korea · false 1,182 / South Korea · true 28
SELECT name, removed, COUNT(*) AS 행수
FROM countries
WHERE name ILIKE '%korea%'
GROUP BY name, removed
ORDER BY 행수 DESC;

### 1단계 — 분석 대상 고르기

**질문** 제2형 당뇨병을 대상으로 한 임상시험은 몇 건인가?

**확인** 한 시험에 같은 질환이 두 번 이상 붙어 있어도 IN 서브쿼리는 1건으로 셈. 이 쿼리가 출발점. 다음 단계부터 여기에 줄을 더해 감.

In [ ]:
%%sql
-- [8번] 1단계 → 1,353
SELECT COUNT(*) AS 시험수
FROM studies
WHERE nct_id IN (SELECT nct_id FROM browse_conditions
                 WHERE mesh_term = 'Diabetes Mellitus, Type 2');

### 2단계 — 값 살펴보기

**질문** 대상 1,353건을 임상 단계별로 묶으면 빈 값이 보이나?

**찾은 것** phase 가 빈 칸(NULL)인 시험이 242건. 'NA'(해당 없음) 611건은 빈 값이 아니라 값임. 인원_있음 = 시험수이므로 등록 인원(enrollment)은 빈 값이 없음.

In [ ]:
%%sql
-- [9번] 2단계 → 9행 (빈 칸 None 242 · NA 611 · 합계 1,353)
SELECT phase AS 단계,
       COUNT(*) AS 시험수,
       COUNT(enrollment) AS 인원_있음
FROM studies
WHERE nct_id IN (SELECT nct_id FROM browse_conditions
                 WHERE mesh_term = 'Diabetes Mellitus, Type 2')
GROUP BY phase
ORDER BY phase;

### 3단계 — 빈 값 거르기

**질문** 단계(phase)가 빈 칸인 시험을 빼면 몇 건이 남나?

**왜 빼나** 단계별로 비교할 때 단계가 빈 시험은 어느 그룹에도 넣을 수 없음. 1,353건 → 1,111건

**'NA'는?** NA(해당 없음, 611건)는 빈 값이 아니라 값이라서 IS NOT NULL로 걸러지지 않음. 남김.

**검산** 1,353 − 242 = 1,111. 표의 시험수를 모두 더해도 1,111이 되어야 함

In [ ]:
%%sql
-- [10번] 3단계 → 8행 (None 행이 빠짐)
SELECT phase AS 단계,
       COUNT(*) AS 시험수,
       COUNT(enrollment) AS 인원_있음
FROM studies
WHERE nct_id IN (SELECT nct_id FROM browse_conditions
                 WHERE mesh_term = 'Diabetes Mellitus, Type 2')
  AND phase IS NOT NULL
GROUP BY phase
ORDER BY phase;

In [ ]:
%%sql
-- [11번] 검산 → 1,111
SELECT COUNT(*) AS 시험수
FROM studies
WHERE nct_id IN (SELECT nct_id FROM browse_conditions
                 WHERE mesh_term = 'Diabetes Mellitus, Type 2')
  AND phase IS NOT NULL;

### 4단계 — 기술통계표(표 1) 만들기

**질문** 임상 단계별로 시험 수와 등록 인원의 평균(SD) · 중앙값(IQR)은?

**확인** 평균이 중앙값보다 훨씬 크면 오른쪽으로 치우친 분포 → 중앙값(Q1–Q3)으로 보고합니다.

**이상값** NA의 표준편차(4,047)가 평균(445)의 9배 → 몇몇 아주 큰 시험이 평균을 끌어올림

In [ ]:
%%sql
-- [12번] 4단계 → 8행 (PHASE3 중앙값 321, PHASE1 32)
SELECT phase AS 단계,
       COUNT(*) AS 시험수,
       COUNT(enrollment) AS 인원_있음,
       ROUND(AVG(enrollment)) AS 평균,
       ROUND(STDDEV(enrollment)) AS 표준편차,
       MEDIAN(enrollment) AS 중앙값,
       QUANTILE_CONT(enrollment, 0.25) AS Q1,
       QUANTILE_CONT(enrollment, 0.75) AS Q3
FROM studies
WHERE nct_id IN (SELECT nct_id FROM browse_conditions
                 WHERE mesh_term = 'Diabetes Mellitus, Type 2')
  AND phase IS NOT NULL
GROUP BY phase
ORDER BY phase;

### 5단계 — 비율 구하기

**질문** 임상 단계별로 등록 인원 500명 이상인 대규모 시험의 비율은?

**분모** 각 단계의 시험 수 = 분모, ‘500명 이상’ = 분자. 분모를 먼저 말로 정합니다.

**다음** 한국 참여 시험만 골라 같은 비율을 비교함 (6단계)

In [ ]:
%%sql
-- [13번] 5단계 → 8행 (PHASE3 대규모 45건 · 37.2%)
SELECT phase AS 단계,
       COUNT(*) AS 시험수,
       COUNT(enrollment) AS 인원_있음,
       -- … 4단계의 평균 · SD · 중앙값 · Q1 · Q3 줄 그대로 …
       COUNT(CASE WHEN enrollment >= 500 THEN 1 END) AS 대규모,
       ROUND(100.0 * COUNT(CASE WHEN enrollment >= 500
             THEN 1 END) / COUNT(*), 1) AS 대규모_비율
FROM studies
WHERE nct_id IN (SELECT nct_id FROM browse_conditions
                 WHERE mesh_term = 'Diabetes Mellitus, Type 2')
  AND phase IS NOT NULL
GROUP BY phase
ORDER BY phase;

### 6단계 — 한국 참여 시험과 비교하기

**질문** 한국이 참여한 시험만 보면 단계별 대규모 비율이 다른가?

**비교** 한국 참여 vs 비참여로 비교함. 비참여는 새로 넣은 줄의 IN을 NOT IN으로 바꿔 실행

**검정** 한국 쪽 건수가 작으므로(단계별 5~22건) 카이제곱 대신 피셔 정확검정으로 판단

In [ ]:
%%sql
-- [14번] 6단계 한국 참여 → 5행 (PHASE2 5건 중 2건 · 40.0%)
SELECT phase AS 단계,
       COUNT(*) AS 시험수,
       COUNT(enrollment) AS 인원_있음,
       -- … 4단계의 평균 · SD · 중앙값 · Q1 · Q3 줄 그대로 …
       COUNT(CASE WHEN enrollment >= 500 THEN 1 END) AS 대규모,
       ROUND(100.0 * COUNT(CASE WHEN enrollment >= 500
             THEN 1 END) / COUNT(*), 1) AS 대규모_비율
FROM studies
WHERE nct_id IN (SELECT nct_id FROM browse_conditions
                 WHERE mesh_term = 'Diabetes Mellitus, Type 2')
  AND phase IS NOT NULL
  AND nct_id IN (SELECT nct_id FROM countries
                 WHERE name = 'South Korea' AND removed = false)
GROUP BY phase
ORDER BY phase;

In [ ]:
%%sql
-- [15번] 비참여 (IN → NOT IN) → 8행 (PHASE2 83건 중 0건 · 0.0%)
SELECT phase AS 단계,
       COUNT(*) AS 시험수,
       COUNT(enrollment) AS 인원_있음,
       -- … 4단계의 평균 · SD · 중앙값 · Q1 · Q3 줄 그대로 …
       COUNT(CASE WHEN enrollment >= 500 THEN 1 END) AS 대규모,
       ROUND(100.0 * COUNT(CASE WHEN enrollment >= 500
             THEN 1 END) / COUNT(*), 1) AS 대규모_비율
FROM studies
WHERE nct_id IN (SELECT nct_id FROM browse_conditions
                 WHERE mesh_term = 'Diabetes Mellitus, Type 2')
  AND phase IS NOT NULL
  AND nct_id NOT IN (SELECT nct_id FROM countries
                 WHERE name = 'South Korea' AND removed = false)
GROUP BY phase
ORDER BY phase;

### 7단계 — 해석하고 보고하기

**표 1** 임상 단계별 등록 인원 (4단계 결과 일부)

| 단계 | 시험수 | 중앙값 (Q1–Q3) |
|---|---:|---|
| PHASE1 | 115 | 32 (24–50) |
| PHASE2 | 88 | 79.5 (30.75–177.25) |
| PHASE3 | 121 | 321 (166–608) |
| PHASE4 | 147 | 88 (30–192) |
| NA | 611 | 60 (30–133.5) |

**해석** 3상 시험의 등록 인원 중앙값은 321명으로, 1상(32명)의 약 10배입니다.

**보고서에 함께 적을 것**

- 대상: mesh_term = 제2형 당뇨병, 1,353건
- 제외: phase 빈 값 242건 → 1,111건 (3단계)
- 중복: IN 서브쿼리라 한 행 = 시험 1건 (1단계)
- 통계: 중앙값(Q1–Q3) · 대규모 기준 500명 (4 · 5단계)
- 비교: 한국 참여 vs 비참여, 피셔 정확검정 (6단계)
- 기준: 자료 추출일 2026-09-19

**다음 단계** CSV로 내보내 R · Python · SPSS에서 분석합니다.

In [ ]:
%%sql
-- [16번] (선택) CSV로 내보내기 → 왼쪽 파일 목록에 대상.csv (3단계 조건, 1,111건)
COPY (SELECT *
      FROM studies
      WHERE nct_id IN (SELECT nct_id FROM browse_conditions
                       WHERE mesh_term = 'Diabetes Mellitus, Type 2')
        AND phase IS NOT NULL)
TO '대상.csv' (HEADER);

### 잠깐 생각해 봅시다

분석표의 행 수가 시험 수보다 많다면 무엇을 의심해야 할까요?

→ JOIN 때문에 같은 시험이 여러 행으로 붙은 것입니다. DISTINCT를 빠뜨렸는지 확인합니다.

## 3. 병원에서 활용하는 SQL 예제

| 누가 | 무엇을 | 어떤 SQL이 쓰이나 |
|---|---|---|
| QI · 적정성평가 담당 | 지표별 대상자 수와 충족률 | 조건으로 대상자 선별 → 분모 · 분자 각각 COUNT |
| 감염관리팀 | 월별 감염 발생 건수, 재원일수 | GROUP BY 월 · 병동, 날짜 차이 계산 |
| 간호부 | 간호 인력 대비 환자 수, 낙상 건수 | GROUP BY 병동 · 근무조, COUNT와 비율 |
| 연구자 | 연구 대상자 선별, 변수 추출 | 여러 표를 JOIN하여 코호트 추출 |
| 원무 · 경영 | 진료과별 실적, 수납 현황 | GROUP BY 진료과 · 기간, SUM과 AVG |

**대상을 고르고(WHERE) → 묶어서 세고(GROUP BY) → 검산한다**

## 4. 검산 — 결과를 믿기 전에

| 확인할 것 | 무엇을 보는가 | 어떻게 |
|---|---|---|
| ① 전체 건수 | 조건 없이 센 값과 비교했을 때 말이 되는가 | `COUNT(*) FROM studies` |
| ② 세는 단위 | 시험 수인가, 행 수인가 | `COUNT(*)` 인가 `COUNT(DISTINCT nct_id)` 인가 |
| ③ 분모 | JOIN 때문에 대상이 줄지 않았는가 | `LEFT JOIN` 으로 다시 세기 |
| ④ 빠진 값 | NULL이 많은 열을 조건에 썼는가 | `COUNT(열)` 로 값 있는 행 세기 |
| ⑤ 이상한 값 | 최소 · 최대가 상식적인가 | `MIN()`, `MAX()` 로 확인 |

In [ ]:
%%sql
-- [17번] ⑤ 이상한 값 → 최소 0명, 최대 4,000,000명. 상식적인가?
SELECT MIN(enrollment) AS 최소인원,
       MAX(enrollment) AS 최대인원
FROM studies
WHERE nct_id IN (SELECT nct_id FROM browse_conditions
                 WHERE mesh_term = 'Diabetes Mellitus, Type 2');

## 5. 과제 1 — 제출 안내

20점 · 노트북 파일(`과제1.ipynb`)을 LMS 에 제출 · **제출 기한 2026-11-10**

| 문항 유형 | 개수 | 내용 |
|---|---|---|
| ① 조회 기초 | 6문항 | SELECT · WHERE · ORDER BY 의 잘못된 SQL 고치기 (7주차) |
| ② 집계와 GROUP BY | 3문항 | COUNT · AVG · GROUP BY 의 잘못된 SQL 고치기 (8주차) |
| ③ 당뇨병 시험 분석 | 3문항 | JOIN 으로 두 표를 연결해 세기 |
| ④ 결과 해석 | 2문항 | 숫자가 뜻하는 바를 문장으로 |

## [실습] 직접 해보기 — 위암으로 연습

질환만 위암(`'Stomach Neoplasms'`)으로 바꾼 같은 단계 SQL 입니다. 빈칸 `___` 하나를 채워 실행하면 결과 아래에 ✅ / ❌ 가 나옵니다.

**열 이름 찾기** — 0단계처럼 `DESCRIBE` 로 표의 열을 봅니다. 표 이름만 바꾸면 다른 표도 볼 수 있습니다.

In [ ]:
%%sql
-- [18번] browse_conditions 의 열 → nct_id (시험 번호) · mesh_term (질환 이름) …
SELECT * FROM (DESCRIBE browse_conditions);

**질환 이름 찾기** — 0단계처럼 `ILIKE` 로 위암의 정확한 이름을 찾습니다.

In [ ]:
%%sql
-- [19번] 위암 이름 찾기 → Stomach Diseases 229 (상위 용어) · Stomach Neoplasms 196 · Stomach Ulcer 3
SELECT mesh_term, COUNT(*) AS 시험수
FROM browse_conditions
WHERE mesh_term ILIKE '%stomach%'
GROUP BY mesh_term
ORDER BY 시험수 DESC;

**Q1. 1단계** — 위암 대상 시험이 몇 건인지 세세요.

In [ ]:
%%sql
-- [20번] Q1
SELECT COUNT(*) AS 시험수
FROM studies
WHERE nct_id ___ (SELECT nct_id FROM browse_conditions
                 WHERE mesh_term = 'Stomach Neoplasms');

**Q2. 2단계** — 위암 시험을 임상 단계(`phase`)별로 묶어 세세요.

In [ ]:
%%sql
-- [21번] Q2
SELECT phase AS 단계,
       COUNT(*) AS 시험수,
       COUNT(enrollment) AS 인원_있음
FROM studies
WHERE nct_id IN (SELECT nct_id FROM browse_conditions
                 WHERE mesh_term = 'Stomach Neoplasms')
GROUP BY ___
ORDER BY phase;

**Q3. 3단계** — 단계가 빈 칸(NULL)인 시험을 빼세요.

In [ ]:
%%sql
-- [22번] Q3
SELECT phase AS 단계,
       COUNT(*) AS 시험수,
       COUNT(enrollment) AS 인원_있음
FROM studies
WHERE nct_id IN (SELECT nct_id FROM browse_conditions
                 WHERE mesh_term = 'Stomach Neoplasms')
  AND phase ___
GROUP BY phase
ORDER BY phase;

**Q4. 4단계** — 단계별 등록 인원의 중앙값을 구하세요.

In [ ]:
%%sql
-- [23번] Q4
SELECT phase AS 단계,
       COUNT(*) AS 시험수,
       ___(enrollment) AS 중앙값
FROM studies
WHERE nct_id IN (SELECT nct_id FROM browse_conditions
                 WHERE mesh_term = 'Stomach Neoplasms')
  AND phase IS NOT NULL
GROUP BY phase
ORDER BY phase;

**Q5. 5단계** — 단계별로 등록 인원 500명 이상인 대규모 시험 수를 세세요.

쓰는 열: `enrollment` (등록 인원)

In [ ]:
%%sql
-- [24번] Q5
SELECT phase AS 단계,
       COUNT(*) AS 시험수,
       COUNT(CASE WHEN ___ THEN 1 END) AS 대규모
FROM studies
WHERE nct_id IN (SELECT nct_id FROM browse_conditions
                 WHERE mesh_term = 'Stomach Neoplasms')
  AND phase IS NOT NULL
GROUP BY phase
ORDER BY phase;

In [ ]:
%%sql
-- [25번] countries 의 열 → nct_id · name (나라 이름) · removed (빠진 국가 표시)
SELECT * FROM (DESCRIBE countries);

**Q6. 6단계** — 한국(`South Korea`)이 참여한 위암 시험만 보세요. 나라 이름을 채웁니다.

In [ ]:
%%sql
-- [26번] Q6
SELECT phase AS 단계,
       COUNT(*) AS 시험수,
       COUNT(CASE WHEN enrollment >= 500 THEN 1 END) AS 대규모
FROM studies
WHERE nct_id IN (SELECT nct_id FROM browse_conditions
                 WHERE mesh_term = 'Stomach Neoplasms')
  AND phase IS NOT NULL
  AND nct_id IN (SELECT nct_id FROM countries
                 WHERE name = ___ AND removed = false)
GROUP BY phase
ORDER BY phase;

## 6. 정리 — 오늘 배운 것

- **준비**: 표 · 열을 파악하고, 대상을 고르고, 빈 값 · 값의 종류를 확인한다
- **정리**: 빈 값을 빼고, DISTINCT로 한 행 = 시험 1건인 분석표를 만든다
- **분석**: 분석표에서 표 1(평균 · 중앙값)과 비율을 낸다
- **보고**: 빼고 정리한 과정을 숫자와 함께 적고, 검산 뒤에 내보낸다

아래 셀로 오늘 문항을 확인합니다.

In [ ]:
진행확인()   # [27번]